In [1]:
import pandas as pd
import re
import string
from collections import defaultdict


from nltk.corpus import stopwords
import stanza
import gensim
from gensim.models.phrases import Phraser, Phrases
from gensim import corpora

pd.set_option('display.max_colwidth', None)

In [11]:
!git Text_analytics_project_MP.ipynb
!git commit -m "Updated Jupyter notebook with new analysis"
!git push origin marco

In [21]:
git add .
git commit -m "Your commit message"

On branch marco
Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   Text_analytics_project_MP.ipynb

no changes added to commit (use "git add" and/or "git commit -a")


In [23]:
!git commit --m "Replaced nltk with Stanza"

On branch marco
Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   Text_analytics_project_MP.ipynb

no changes added to commit (use "git add" and/or "git commit -a")


# Data preparation

In [143]:
## Columns resetting
df = pd.read_table('data/haspeede2_dev/haspeede2_dev_taskAB.tsv', header= 0)
df = df.drop('stereotype', axis = 1)
# Rename the column 'text ', there was an extraspace
df.rename(columns = {'text ': 'text'}, inplace = True)

In [145]:
def clean_df(df):
    # Lowercase tweets
    df['text'] = df['text'].apply(lambda x: x.lower())
    custom_punctuation = string.punctuation + "’‘" + '’' + '' + '``' + "''"
    # Preprocess text in a single step
    df['text_processed'] = df['text'].apply(lambda x: re.sub(r'@\w+', '', x))  # Remove mentions
    df['text_processed'] = df['text_processed'].apply(lambda x: re.sub(r'\.{2,}', '', x))  # Remove multiple dots
    df['text_processed'] = df['text_processed'].apply(lambda x: re.sub(r'\d+', '', x))  # Remove numbers
    df['text_processed'] = df['text_processed'].apply(lambda x: re.sub(r'#', '', x))  # Remove hashtags but keep text
    df['text_processed'] = df['text_processed'].apply(lambda x: re.sub(r'url', '', x))  # Remove URLs
    df['text_processed'] = df['text_processed'].apply(lambda x: re.sub(r'\s+', ' ', x).strip())  # Remove extra space

    # Initialize Stanza pipeline with tokenization
    nlp = stanza.Pipeline(lang='it', processors='tokenize')  # Use appropriate language code

    # Apply Stanza tokenization to each text in the DataFrame
    df['word_token'] = df['text_processed'].apply(lambda text: [word.text for sent in nlp(text).sentences for word in sent.words])

    # Remove stopwords and punctuation
    df['tokenized'] = [[word for word in set(tweet) 
         if word in word not in stopwords.words('italian') and word not in custom_punctuation]  # Remove punctuation and stopwords
        for tweet in df['word_token']]

    # Tokenize into sentences
    #df['sentence_token'] = df['text_processed'].apply(lambda text: [sent.text for sent in nlp(text).sentences])
    # Drop intermediate columns
    df = df.drop('word_token', axis=1)
    
    return df

In [153]:
df = clean_df(df)
# n° of tokens per tweet
df['n_token'] = df['tokenized'].apply(len)
# n° of sentences per tweet
#df['n_sentence'] = df['sentence_token'].apply(len)

## Badwords

In [155]:
file = open('data/lista_badwords.txt')
badwords = file.readlines()
badwords = [word.replace('\n', '') for word in badwords]

# How many badword per tweet?
badword_0_1 = []
for l in df['tokenized']:
    counter = 0 
    for word in l:
        if word in badwords:
            counter += 1
    badword_0_1.append(counter)
df['badword'] = badword_0_1

# Creating two corpora of tweets as str
- token0: tokenized and cleaned corpus of not hs
- token0: tokenized and cleaned corpus of hs
- sent: list of lists of tokenized tweets

In [157]:
token0, token1, sent = '', '', []
custom_punctuation = string.punctuation + "’‘" + '’' + '' + '``' + "''" + '“' + '»' + '«' + '”' + '-'

for tweet, label in zip(df['text_processed'], df['hs']):
    tweet_ = tweet 
    tweet_ = word_tokenize(tweet_)
    tweet_ = [word for word in tweet_ if word not in stopwords.words('italian') and word not in custom_punctuation]
    
    if label == 0:
        token0 += tweet
    else:
        token1 += tweet
    sent.append(tweet_)


token0 = word_tokenize(token0)
token1 = word_tokenize(token1)
token0 = [word for word in token0 if word not in stopwords.words('italian') and word not in custom_punctuation]
token1 = [word for word in token1 if word not in stopwords.words('italian') and word not in custom_punctuation]

In [158]:
freq_dist0 = nltk.FreqDist(token0)
freq_dist1 = nltk.FreqDist(token1)

print(freq_dist0.most_common(10))
print('\n')
print(freq_dist1.most_common(10))

[('rom', 1086), ('migranti', 725), ('immigrati', 399), ('italia', 335), ('roma', 300), ('campo', 289), ('italiani', 233), ('stranieri', 223), ('nomadi', 222), ('solo', 214)]


[('immigrati', 509), ('migranti', 472), ('rom', 335), ('italiani', 322), ('italia', 294), ('casa', 226), ('stranieri', 204), ('clandestini', 201), ('solo', 187), ('islamici', 182)]


In [163]:
# Create a list of list, where each sublist contains the tokenized version of the tweet
sents = [l for l in df['tokenized']]

## Collocations

In [167]:
# The number as expected decreased, as in the first attempt the pre-processing 
#was less involving and the result of this code was: (129277, 6837); (111548, 6837)--> using NLTK, (91521, 6837) --> using stanza 

len(Phrases(sents).vocab), len(sents)

(91521, 6837)

In [169]:
bigram = Phraser(Phrases(sents))

In [171]:
collocations = list()
for key, score in bigram.phrasegrams.items():
    collocations.append((key,score))
bigrams_df = pd.DataFrame(collocations,columns = ['collocation', 'score'])
bigrams_df.sort_values(by=['score'],ascending=False)

,collocation,score
30,asilo_richiedenti,1198.489286
79,global_compact,1056.011538
1,studentessa_cinese,743.785217
2,cinese_morta,593.125098
80,compact_global,528.005769
...,...,...
50,”_l’,10.336684
61,salvini_può,10.271717
52,vuole_immigrati,10.268526
68,immigrati_delinquenti,10.024037


In [189]:
stopword_list = stopwords.words('italian')
sents_w_bigram = bigram[sents]
trigrams = Phraser(Phrases(sents_w_bigram, connector_words=stopword_list))

In [193]:
collocations = list()
for key, score in trigrams.phrasegrams.items():
    collocations.append((key,score))
trigrams_df = pd.DataFrame(collocations,columns = ['collocation', 'score'])
trigrams_df.sort_values(by=['score'],ascending=False)

,collocation,score
88,global_compact,79023.428571
89,compact_global,39511.714286
2,studentessa_cinese,9308.048077
34,asilo_richiedenti,8451.116667
25,cinese_morta,2786.082418
...,...,...
49,italia_meglio,10.445255
6,–_campo,10.289509
42,ora_clandestini,10.243453
38,casa_”,10.023266


In [195]:
collocations_df = pd.concat([bigrams_df, trigrams_df])
collocations_df = collocations_df.drop_duplicates(subset = ["collocation"], keep='last')
collocations_df['ngram'] = collocations_df['collocation'].apply(lambda x: 'bigram' if x.count('_') == 1 else 'ngram')
collocations_df[collocations_df['ngram'] == 'ngram'].sort_values('score',ascending=False)

,collocation,score,ngram
11,studentessa_cinese_morta_roma,344.007463,ngram
26,cinese_morta_roma,52.924225,ngram
1,campi_rom_mafia,15.314618,ngram
22,processo_campi_rom,13.801497,ngram


# Working again on the DF
- Lemmatization

In [203]:
# Initialize the Stanza pipeline for Italian language
nlp = stanza.Pipeline('it', processors='tokenize, mwt,pos,lemma')

# Apply lemmatization with Stanza
df['lemmatized_txt'] = [[word.lemma for word in nlp(' '.join(words_list)).sentences[0].words] for words_list in df['tokenized']]

2024-11-25 10:45:18 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


2024-11-25 10:45:18 INFO: Downloaded file to C:\Users\marco\stanza_resources\resources.json
2024-11-25 10:45:18 INFO: Loading these models for language: it (Italian):
| Processor | Package           |
---------------------------------
| tokenize  | combined          |
| mwt       | combined          |
| pos       | combined_charlm   |
| lemma     | combined_nocharlm |

2024-11-25 10:45:18 INFO: Using device: cpu
2024-11-25 10:45:18 INFO: Loading: tokenize
C:\Users\marco\anaconda4\Lib\site-packages\stanza\models\tokenization\trainer.py:82: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the function

In [207]:
df.to_csv('data/df_cleaned_mp.csv', index=False) # commited on GitHub

In [235]:
# Process the lemmatized text in the DataFrame
for element in df['lemmatized_txt']:
    pos_inner_num = []
    pos_inner_str = []
    
    # Process each text element as a full sentence (using Stanza)
    doc = nlp(' '.join(element))  # Combine tokens into a single string
    
    # Iterate over each word in the processed document (Sentence object)
    for sent in doc.sentences:  # Iterate through sentences (Stanza's output is sentence-based)
        for word in sent.words:  # Each word in the sentence
            pos_inner_num.append(word.pos)  # Numeric POS tag
            pos_inner_str.append(word.xpos)  # String representation of the POS tag
            
    # Add the results for the current element (sentence or tokenized text)
    pos_num.append(pos_inner_num)
    pos_str.append(pos_inner_str)

In [241]:
final = []
for l in pos_str:
    dic = {}
    for element in l:
        dic[element] = dic.get(element, 0)+1
    final.append(dic)

In [243]:
df_pos = pd.DataFrame(final)
df_pos = df_pos.fillna(0).astype(int)

In [245]:
df = pd.concat([df,df_pos], axis = 1)

In [136]:
## I took the next pieces of code from the notebook 6.5

In [251]:
# list of 1 & 2) tokenized words without common words
sentences = []
for l in df['lemmatized_txt']:
    sentences.append(l)
# remove words that appear only 1 
frequency = defaultdict(int)
for sentence in sentences:
    for token in sentence:
        frequency[token] += 1
texts = [[token for token in sentence if frequency[token] > 1]for sentence in sentences]

In [253]:
COUNTER_no_rare_words, COUNTER_with_rare_words= 0, 0 
for l in texts:
    for word in l:
        COUNTER_no_rare_words +=1
        
for l in sentences:
    for word in l:
        COUNTER_with_rare_words +=1
        
print('Without rare word:', COUNTER_no_rare_words,'\n' 'With rare words:', COUNTER_with_rare_words)

Without rare word: 83357 
With rare words: 91070


In [255]:
dictionary = corpora.Dictionary(texts)
corpus = [dictionary.doc2bow(text) for text in texts]
tfidf = gensim.models.TfidfModel(corpus, smartirs='ntc')
word_doc_matrix_tfidf = gensim.matutils.corpus2dense(tfidf[corpus],num_terms = len(dictionary))

#### Creating the dataset with token (as features) and score (are records)

In [257]:
df1 = pd.DataFrame(word_doc_matrix_tfidf, columns = [s for s in df['text']], index = [k for k in dictionary.token2id.keys()])
df1 = df1.T

In [259]:
df1_reset = df1.reset_index(drop=True)
df = pd.concat([df, df1_reset], axis=1)

# ML

In [295]:
import pandas as pd
import numpy as np
import matplotlib.pylab as plt 
import seaborn as sns
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import train_test_split, cross_val_score 
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split, cross_val_score 

from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.linear_model import LogisticRegression

from sklearn.feature_extraction.text import TfidfTransformer, CountVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC # Linear Support Vector Classifier
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import RandomizedSearchCV

In [279]:
dfml = df.drop(['id', 'text', 'tokenized', 'text_processed', 'lemmatized_txt'], axis = 1)
X = dfml.drop('hs', axis=1).values
y = dfml['hs'].values

In [287]:
numeric_columns = ['n_token', 'badword']
df_text = dfml.drop(columns = ['n_token', 'badword', 'hs'])
text_columns = df_text.columns

preprocessor = ColumnTransformer(
    transformers=[('text', Pipeline([
            ('tfidf', TfidfTransformer()),  # Apply TF-IDF to text columns
            ('sel', SelectKBest(chi2, k=5000))  # Feature selection after TF-IDF
        ]), text_columns),  ('num', StandardScaler(), numeric_columns)])

X = dfml.drop(columns=['hs'])  # Adjust with your target column name
y = dfml['hs']  # Adjust with your target column name
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.02, random_state=42, stratify=y)

##### Classification with StandardScaler worsens
##### This is the result of a classification with

* punctuation
* bad_words count
* n_of token and sentences
* Count of pos --> without this the classification worsen (Macro AVG F1 0.68)

In [301]:
bin_pipeline = Pipeline([('preprocessor', preprocessor), 
                         ('learner', LogisticRegression(random_state=42, max_iter=500, verbose = 1, solver = 'saga',penalty = 'l1'))])
# Hyperparameter grid to search over
param_dist = {
    'learner__C': np.logspace(-4, 4, 20),  # Regularization strength (inverse)
    'learner__solver': ['saga', 'liblinear'],  # Solvers to test
    'learner__penalty': ['l1', 'l2'],  # Penalty terms
    'learner__max_iter': [100, 200, 500],  # Maximum number of iterations
}

# Set up RandomizedSearchCV with 5-fold cross-validation
random_search = RandomizedSearchCV(
    bin_pipeline, param_distributions=param_dist, 
    n_iter=10,  # Number of random combinations to test
    scoring='f1',  # Metric to optimize (could also use 'f1', 'roc_auc', etc.)
    cv=5,  # Cross-validation folds
    verbose=1,  # Print progress
    random_state=42,  # For reproducibility
    n_jobs=-1  # Use all cores available
)

# Fit the model
random_search.fit(X_train, y_train)

# Print the best hyperparameters found
print("Best hyperparameters found:", random_search.best_params_)

# Predict using the best model
bin_predictions = random_search.best_estimator_.predict(X_test)

print(classification_report(y_test, bin_predictions,zero_division=0))

Fitting 5 folds for each of 10 candidates, totalling 50 fits
max_iter reached after 13 seconds
Best hyperparameters found: {'learner__solver': 'saga', 'learner__penalty': 'l1', 'learner__max_iter': 200, 'learner__C': 78.47599703514607}
              precision    recall  f1-score   support

           0       0.79      0.79      0.79        82
           1       0.69      0.69      0.69        55

    accuracy                           0.75       137
   macro avg       0.74      0.74      0.74       137
weighted avg       0.75      0.75      0.75       137



C:\Users\marco\anaconda4\Lib\site-packages\sklearn\linear_model\_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
